# Embedding Model Comparison — Identity Similarity

Compares three sentence-transformer models on their ability to capture mechanic similarity between Limbus Company identities.

**Candidates:**
- `all-MiniLM-L6-v2` — 22M params, 384-dim, fastest
- `all-mpnet-base-v2` — 110M params, 768-dim, highest quality
- `bge-small-en-v1.5` — retrieval-optimized, 384-dim

**Test data:** Skill and passive descriptions from `docs/parsed-ids/*.md`

**Success criterion:** Mechanically similar identities (e.g., both Bleed-focused) should have higher cosine similarity than dissimilar ones (e.g., Bleed vs. Poise).

In [ ]:
# pip install sentence-transformers scikit-learn matplotlib

import re
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

## 1. Extract skill/passive descriptions from parsed identities

In [ ]:
PARSED_IDS_DIR = Path("../docs/parsed-ids")


def extract_descriptions(md_text: str) -> str:
    """Extract skill effects and passive text from a parsed identity markdown file.
    
    Strips headers, stat tables, and metadata — keeps effect descriptions,
    coin effects, passive text, and support passive text.
    """
    lines = md_text.splitlines()
    description_lines = []
    for line in lines:
        stripped = line.strip()
        # Skip markdown headers, table separators, empty lines, and metadata
        if not stripped:
            continue
        if stripped.startswith("#"):
            continue
        if stripped.startswith("|") and all(c in "|-: " for c in stripped):
            continue
        if stripped.startswith("---"):
            continue
        if stripped.startswith(">"):  # flavor quote
            continue
        if stripped.startswith("**Rarity") or stripped.startswith("**Season") or stripped.startswith("**Release") or stripped.startswith("**Traits"):
            continue
        if stripped.startswith("**Stagger"):
            continue
        # Keep stat tables with actual game data
        if stripped.startswith("|") and ("Offense Level" in stripped or "HP" in stripped or "Coin |" in stripped):
            continue
        # Keep effect descriptions, [On Use], passive text, etc.
        description_lines.append(stripped)

    return " ".join(description_lines)


identity_descriptions = {}
for md_file in sorted(PARSED_IDS_DIR.glob("*.md")):
    text = md_file.read_text(encoding="utf-8")
    identity_descriptions[md_file.stem] = extract_descriptions(text)

for name, desc in identity_descriptions.items():
    print(f"{name}: {len(desc)} chars")
    print(f"  Preview: {desc[:150]}...\n")

## 2. Encode with each model and compute similarity matrices

In [ ]:
MODEL_NAMES = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "BAAI/bge-small-en-v1.5",
]

identity_names = list(identity_descriptions.keys())
texts = [identity_descriptions[name] for name in identity_names]

results = {}
for model_name in MODEL_NAMES:
    print(f"\nLoading {model_name}...")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, convert_to_numpy=True)
    sim_matrix = cosine_similarity(embeddings)
    results[model_name] = {
        "embeddings": embeddings,
        "similarity": sim_matrix,
    }
    print(f"  Embedding shape: {embeddings.shape}")
    print(f"  Similarity matrix:")
    for i, name_i in enumerate(identity_names):
        row = "  ".join(f"{sim_matrix[i][j]:.3f}" for j in range(len(identity_names)))
        print(f"    {name_i:40s} {row}")

## 3. Visualize similarity matrices

In [ ]:
short_names = [n.replace("_", " ") for n in identity_names]

fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(6 * len(MODEL_NAMES), 5))

for idx, model_name in enumerate(MODEL_NAMES):
    ax = axes[idx]
    sim = results[model_name]["similarity"]
    im = ax.imshow(sim, cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(len(short_names)))
    ax.set_yticks(range(len(short_names)))
    ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(short_names, fontsize=7)
    ax.set_title(model_name.split("/")[-1], fontsize=10)

    for i in range(len(identity_names)):
        for j in range(len(identity_names)):
            ax.text(j, i, f"{sim[i][j]:.2f}", ha="center", va="center", fontsize=8)

fig.suptitle("Identity Similarity — Model Comparison", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Expected-pair analysis

Ring Pointillist Student Yi Sang and Ring Apprentice Faust both focus on **Bleed**, so they should be more similar to each other than to Blade Lineage Salsu Yi Sang (Poise-focused).

In [ ]:
EXPECTED_SIMILAR = ("Ring_Pointillist_Student_Yi_Sang", "Ring_Apprentice_Faust")
EXPECTED_DISSIMILAR = ("Blade_Lineage_Salsu_Yi_Sang", "Ring_Apprentice_Faust")

idx_map = {name: i for i, name in enumerate(identity_names)}

print(f"{'Model':<35s} {'Bleed↔Bleed':>12s} {'Poise↔Bleed':>12s} {'Delta':>8s} {'Pass?':>6s}")
print("-" * 75)
for model_name in MODEL_NAMES:
    sim = results[model_name]["similarity"]
    sim_similar = sim[idx_map[EXPECTED_SIMILAR[0]]][idx_map[EXPECTED_SIMILAR[1]]]
    sim_dissimilar = sim[idx_map[EXPECTED_DISSIMILAR[0]]][idx_map[EXPECTED_DISSIMILAR[1]]]
    delta = sim_similar - sim_dissimilar
    passed = "YES" if delta > 0 else "NO"
    print(f"{model_name.split('/')[-1]:<35s} {sim_similar:>12.4f} {sim_dissimilar:>12.4f} {delta:>+8.4f} {passed:>6s}")

## 5. Verdict

**Expected finding:** All three models should correctly rank Bleed↔Bleed similarity higher than Poise↔Bleed.

- `all-MiniLM-L6-v2` — fastest, good enough for prototype. Best throughput.
- `all-mpnet-base-v2` — highest semantic quality, but 5x slower.
- `bge-small-en-v1.5` — good retrieval-oriented balance.

**Recommendation for the pipeline:** Start with `all-MiniLM-L6-v2` for speed during development; switch to `all-mpnet-base-v2` or `bge-small-en-v1.5` if similarity quality is insufficient at scale (172 identities).